In [8]:
import verovio
from IPython.display import SVG, HTML, display

mei_path = "Bach_BWV_0772.mei"   # same folder as notebook -> just the filename

# Create toolkit
tk = verovio.toolkit()

# Set rendering options
tk.setOptions({
    "pageWidth": 2000,
    "pageHeight": 1000,
    "scale": 35,
    "adjustPageHeight": True
})

# Load the MEI file
tk.loadFile(mei_path)

# Render MIDI as base64
midi_b64 = tk.renderToMIDI()

# Show a player in the notebook
html = f"""
<script type="module" src="https://cdn.jsdelivr.net/npm/html-midi-player@1.6.0/+esm"></script>
<midi-player src="data:audio/midi;base64,{midi_b64}" sound-font></midi-player>
"""
display(HTML(html))

In [7]:
import re

bpm = 80

with open(mei_path, "r", encoding="utf-8") as f:
    mei_text = f.read()

if 'midi.bpm="' in mei_text:
    mei_text = re.sub(r'midi\.bpm="\d+(\.\d+)?"',
                        f'midi.bpm="{bpm}"',
                        mei_text,
                        count=1)
else:
    mei_text = re.sub(
        r'(<measure\b[^>]*>)',
        rf'\1\n  <tempo midi.bpm="{bpm}">♩ = {bpm}</tempo>',
        mei_text,
        count=1
    )

tk = verovio.toolkit()

# Set rendering options
tk.setOptions({
    "pageWidth": 2000,
    "pageHeight": 1000,
    "scale": 35,
    "adjustPageHeight": True
})

# Load the MEI file
tk.loadData(mei_text)

# Render MIDI as base64
midi_b64 = tk.renderToMIDI()

# Show a player in the notebook
html = f"""
<script type="module" src="https://cdn.jsdelivr.net/npm/html-midi-player@1.6.0/+esm"></script>
<midi-player src="data:audio/midi;base64,{midi_b64}" sound-font></midi-player>
"""
display(HTML(html))

In [15]:
import re
import verovio
from IPython.display import HTML, display

def play_mei_excerpt(mei_path, start_q, end_q, bpm=120):
    with open(mei_path, "r", encoding="utf-8") as f:
        mei_text = f.read()

    # Set or inject tempo
    if 'midi.bpm="' in mei_text:
        mei_text = re.sub(
            r'midi\.bpm="\d+(\.\d+)?"',
            f'midi.bpm="{bpm}"',
            mei_text,
            count=1
        )
    else:
        mei_text = re.sub(
            r'(<measure\b[^>]*>)',
            rf'\1\n  <tempo midi.bpm="{bpm}">♩ = {bpm}</tempo>',
            mei_text,
            count=1
        )

    tk = verovio.toolkit()
    tk.loadData(mei_text)
    midi_b64 = tk.renderToMIDI()

    start_sec = start_q * 60 / bpm
    end_sec = end_q * 60 / bpm

    html = f"""
    <script type="module" src="https://cdn.jsdelivr.net/npm/html-midi-player@1.6.0/+esm"></script>

    <div style="margin:10px 0;">
      <midi-player id="player"
                   src="data:audio/midi;base64,{midi_b64}"
                   sound-font>
      </midi-player>
    </div>

    <button id="play_excerpt_btn" disabled>Play excerpt</button>
    <span id="status" style="margin-left:10px;">Loading player...</span>

    <script>
    (function() {{
        const player = document.getElementById("player");
        const btn = document.getElementById("play_excerpt_btn");
        const status = document.getElementById("status");

        const startSec = {start_sec};
        const endSec = {end_sec};

        player.addEventListener("load", () => {{
            btn.disabled = false;
            status.textContent = "Ready";
            console.log("MIDI player loaded");
        }});

        player.addEventListener("start", () => {{
            console.log("Playback started at", player.currentTime);
        }});

        player.addEventListener("stop", () => {{
            console.log("Playback stopped at", player.currentTime);
        }});

        btn.addEventListener("click", async () => {{
            try {{
                status.textContent = "Playing...";
                player.stop();
                player.currentTime = startSec;
                player.start();

                const timer = setInterval(() => {{
                    if (!player.playing || player.currentTime >= endSec) {{
                        player.stop();
                        clearInterval(timer);
                        status.textContent = "Stopped";
                    }}
                }}, 50);
            }} catch (err) {{
                console.error(err);
                status.textContent = "Error: " + err;
            }}
        }});
    }})();
    </script>
    """
    display(HTML(html))

In [19]:
play_mei_excerpt(mei_path, 3.25, 4.25, bpm=60)